# Event Registry: Normalizing Athletic Events

This notebook demonstrates how to use the EventRegistry to normalize event names from various sources (athle.fr, CSVs, APIs) to canonical event IDs.

## Why Normalization?

Athletic event data from different sources uses inconsistent naming:
- French federation (athle.fr): "Longueur", "Hauteur", "Poids"
- World Athletics codes: "LJ", "HJ", "SP"
- English: "Long Jump", "High Jump", "Shot Put"

The EventRegistry provides a single source of truth, mapping all synonyms to canonical event IDs.

In [ ]:
import pandas as pd
from athletics_performance.events import EventRegistry

# Load the event registry
registry = EventRegistry()

print(f"Registry loaded with {len(registry.list_events())} events")
print(f"\nFirst 10 events: {registry.list_events()[:10]}")

## Resolving Event Names

The `resolve()` method handles synonyms across languages and formats.

In [ ]:
# Test various event name formats
test_names = [
    "100m",           # English
    "100 mètres",     # French full
    "longueur",       # French: Long Jump
    "saut en longueur",  # French full: Long Jump
    "LJ",             # World Athletics code
    "long jump",      # English
    "hauteur",        # French: High Jump
    "HJ",             # World Athletics
    "poids",          # French: Shot Put
    "SP",             # World Athletics
]

results = []
for name in test_names:
    canonical = registry.resolve(name)
    results.append({
        'Input': name,
        'Canonical ID': canonical if canonical else 'NOT FOUND'
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

## Event Metadata

Get detailed metadata for any event.

In [ ]:
# Get metadata for selected events
events_to_check = ['100m', 'long_jump', 'shot_put', 'marathon']

metadata_list = []
for event_id in events_to_check:
    metadata = registry.get_metadata(event_id)
    if metadata:
        metadata_list.append({
            'Event ID': event_id,
            'Description': metadata.description,
            'Measurement': metadata.measurement,
            'Unit': metadata.unit,
            'World Athletics': metadata.world_athletics_id,
        })

df_metadata = pd.DataFrame(metadata_list)
print(df_metadata.to_string(index=False))

## Using EventRegistry in Silver Layer

Normalize events during silver layer transformation.

In [ ]:
# Example: Create sample data from multiple sources
sample_data = pd.DataFrame({
    'perf_id': ['p1', 'p2', 'p3', 'p4', 'p5'],
    'athlete_name': ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'event_name': ['longueur', 'hauteur', 'poids', '100m', 'saut triple'],
    'performance': [7.5, 1.95, 18.2, 11.5, 15.3],
    'date': pd.date_range('2025-01-01', periods=5),
})

print("Sample data from multiple sources:")
print(sample_data.to_string())

In [ ]:
# Silver layer transformation: Normalize events
def normalize_events(df):
    """Normalize event names to canonical IDs."""
    df = df.copy()
    
    # Resolve each event name
    df['event_id'] = df['event_name'].apply(
        lambda name: registry.resolve(name)
    )
    
    # Get measurement type for each event
    df['measurement'] = df['event_id'].apply(
        lambda eid: registry.get_measurement(eid) if eid else None
    )
    
    # Flag unrecognized events
    df['recognized'] = df['event_id'].notna()
    
    return df

# Apply transformation
silver_data = normalize_events(sample_data)

print("\nSilver layer data (with normalized events):")
print(silver_data[['athlete_name', 'event_name', 'event_id', 'measurement', 'recognized']].to_string())

## Handling Unrecognized Events

The transformation identifies which events weren't recognized.

In [ ]:
# Show unrecognized events
unrecognized = silver_data[~silver_data['recognized']]

if len(unrecognized) > 0:
    print(f"Found {len(unrecognized)} unrecognized event(s):")
    print(unrecognized[['athlete_name', 'event_name']].to_string())
else:
    print("All events recognized!")

# Show recognized events
print(f"\nRecognized {silver_data['recognized'].sum()} of {len(silver_data)} events")

## Filtering by Measurement Type

Use metadata to filter by measurement type (time vs distance).

In [ ]:
# Filter to only time-based events
time_events = silver_data[silver_data['measurement'] == 'time']
distance_events = silver_data[silver_data['measurement'] == 'distance']

print(f"Time events ({len(time_events)}):")
print(time_events[['athlete_name', 'event_id', 'performance']].to_string())

print(f"\nDistance events ({len(distance_events)}):")
print(distance_events[['athlete_name', 'event_id', 'performance']].to_string())

## Event Statistics

Analyze which events are registered.

In [ ]:
# Get all registered events with their measurement types
all_events = registry.list_events()

event_stats = []
for event_id in all_events:
    metadata = registry.get_metadata(event_id)
    event_stats.append({
        'Event ID': event_id,
        'Measurement': metadata.measurement if metadata else 'unknown',
    })

df_events = pd.DataFrame(event_stats)

print(f"\nTotal events in registry: {len(df_events)}")
print(f"\nEvent distribution by type:")
print(df_events['Measurement'].value_counts())

print(f"\nSample of registered events:")
print(df_events.head(10).to_string(index=False))

## Advanced: Case-Insensitive Matching

The registry handles various cases.

In [ ]:
# Test case insensitivity
test_cases = [
    'longueur',
    'LONGUEUR',
    'Longueur',
    'LoNgUeUr',
]

print("Case-insensitive matching:")
for test in test_cases:
    result = registry.resolve(test)
    print(f"  {test:15} → {result}")

## Key Takeaways

1. **EventRegistry** normalizes event names from multiple sources
2. **resolve()** maps synonyms to canonical IDs (case-insensitive)
3. **Metadata** includes measurement type, unit, and World Athletics codes
4. **Silver layer** is the ideal place to normalize events
5. **Unrecognized events** are flagged for review
6. **Filtering** by measurement type enables type-specific processing

See the documentation for more examples:
- [Event Registry Guide](../docs/guide_event_registry.rst)
- [Medallion Architecture Guide](../docs/guide_medallion.rst)